# Theorem 1: The 2-for-1

> **Based on NBA play-by-play data from 2019–2024, rushing a shot in tied
> games shows a positive win-rate signal around 18–22 seconds remaining.
> The effect is noisy and no single threshold reliably separates when
> rushing helps from when it hurts.**

This notebook reproduces the data collection and visualisation for Theorem 1.
Run each cell in order. Data is saved to `data/processed/theorem1_sweep.csv`
so that plots can be regenerated without re-running the full pipeline.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────────────
import sys
import pathlib

# Make sure the project root is on the path so src.* imports work.
ROOT = pathlib.Path().resolve().parent  # notebooks/ -> project root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib
matplotlib.use("Agg")  # headless; swap to "inline" in a live Jupyter kernel
import matplotlib.pyplot as plt
import numpy as np

PROCESSED_DIR = ROOT / "data" / "processed"
IMAGES_DIR    = ROOT / "docs" / "assets" / "images"

print("Project root  :", ROOT)
print("Processed dir :", PROCESSED_DIR)

## Step 1 – Collect data

Run `collect()` to scan the historical play-by-play log for tied-game
possessions, bucket them by seconds remaining, and save win rates for
the **rush** (shoot immediately) and **normal** (hold the ball) strategies
to `theorem1_sweep.csv`.

> *Skip this cell if `theorem1_sweep.csv` already exists in `data/processed/`.*

In [ ]:
from src.theorems.theorem1 import collect

csv_path = collect(out_dir=PROCESSED_DIR, processed_dir=PROCESSED_DIR)
print("Saved:", csv_path)

## Step 2 – Load sweep data

`load_sweep()` reads the CSV and returns a list of per-second-bucket
dictionaries with `ev_rush`, `ev_normal`, and `ev_gain` fields.

In [ ]:
from src.theorems.theorem1 import load_sweep

sweep = load_sweep(PROCESSED_DIR)

# Preview all rows
print(f"{'seconds':>10} {'ev_rush':>10} {'ev_normal':>10} {'ev_gain':>10} {'optimal':>10}")
print("-" * 55)
for row in sweep:
    print(
        f"{row['seconds_remaining']:>10} "
        f"{row['ev_rush']:>10.4f} "
        f"{row['ev_normal']:>10.4f} "
        f"{row['ev_gain']:>10.4f} "
        f"{str(row['rush_is_optimal']):>10}"
    )

## Step 3 – Reproduce the plot

`plot()` generates the two-panel figure:

* **Top panel** – absolute historical win percentage for Rush vs. Normal.
* **Bottom panel** – win-percentage gain from rushing (green = rushing is
  better, red = normal possession is better).

In [ ]:
from src.theorems.theorem1 import plot

IMAGES_DIR.mkdir(parents=True, exist_ok=True)

fig_path = plot(processed_dir=PROCESSED_DIR, images_dir=IMAGES_DIR)
print("Saved figure:", fig_path)

## Step 4 – Display the plot inline

In [ ]:
from src.theorems.theorem1 import load_sweep, FONT_FAMILY

sweep     = load_sweep(PROCESSED_DIR)
seconds   = [r["seconds_remaining"] for r in sweep]
ev_rush   = [r["ev_rush"]           for r in sweep]
ev_normal = [r["ev_normal"]         for r in sweep]
ev_gain   = [r["ev_gain"]           for r in sweep]

plt.rcParams.update({
    "font.family": FONT_FAMILY,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

ax1 = axes[0]
ax1.plot(seconds, ev_rush,   color="#E63946", linewidth=2.2, label="Rush (shoot now)")
ax1.plot(seconds, ev_normal, color="#457B9D", linewidth=2.2, label="Normal (full possession)")
ax1.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
ax1.set_ylabel("Historical Win Percentage")
ax1.set_title(
    "Theorem 1: The 2-for-1\nHistorical Win Percentage: Rush Shot vs. Full Possession",
    fontweight="bold",
)
ax1.legend(loc="upper right")
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
gain_arr = np.array(ev_gain)
colors   = np.where(gain_arr >= 0, "#2DC653", "#E63946")
ax2.bar(seconds, gain_arr, color=colors, width=1.6, alpha=0.85)
ax2.axhline(0, color="black", linewidth=1.0)
ax2.set_xlabel("Seconds Remaining in Possession")
ax2.set_ylabel("Historical Win % Gain from Rushing (pp)")
ax2.set_title("Win % Gain: Rush − Normal  (green = rushing is better)")
ax2.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

## Step 5 – Generate the Markdown documentation

In [ ]:
from src.theorems.theorem1 import generate_doc

DOCS_DIR = ROOT / "docs"
doc_path = generate_doc(processed_dir=PROCESSED_DIR, docs_dir=DOCS_DIR)
print("Saved doc:", doc_path)